# Complementary Categories

Which categories get bought together. `also_buy` is the signal: for each item
Amazon lists the asins shown as "frequently bought together", so mapping both
ends of every edge to its category turns a product-level co-purchase list into
something a category-by-category complementarity matrix aggregates from.

Two sources, and the split between them matters:

| Source | Grain | Role |
| --- | --- | --- |
| `df_features.pkl` | one row per `asin`, schema categories only | the **source** side of each edge, carrying the extracted features |
| `meta_Home_and_Kitchen_filtered.csv` | one row per `asin`, whole catalogue | the **target** side lookup |

`run_feature_extraction` calls `filter_by_cat_3`, so `df_features.pkl` holds
only items whose `cat_3` is one of the 69 categories in
`master_metadata.json`. `also_buy` points anywhere — other categories, and
outside Home & Kitchen entirely — so target categories are looked up in the
CSV, which was never filtered. Edges whose target is in neither table are kept
with their categories marked `Not in catalogue`: how much of `also_buy` leaves
the catalogue is a finding, not something to drop on the floor.

`cat_4` is folded through `category_taxonomy.json` here, the same whitelist
`ttn.ipynb` §3 applies — the pipeline does not produce `cat_4_clean`. `cat_2`
and `cat_3` have no cleaned variant and are used as parsed.

**`also_buy` must be in the pickle.** It only reaches `df_features.pkl` if the
extraction ran against a CSV that already carried the column — that is, one
rebuilt by `data/variable_selection.ipynb` after `also_buy` was added to
`fields_to_keep`. §1 checks and says so if not.

In [1]:
import ast
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Notebooks run from notebooks/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
FEATURES_PATH = DATA_DIR / "df_features.pkl"
CATALOGUE_PATH = DATA_DIR / "meta_Home_and_Kitchen_filtered.csv"
TAXONOMY_PATH = DATA_DIR / "category_taxonomy.json"

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the extracted features

`df_features.pkl` is the raw metadata with the extraction results already
joined on it, so loading it is what connects the categories and `also_buy` to
the features. Everything else in this notebook works off `asin`.

In [ ]:
df_features = pd.read_pickle(FEATURES_PATH)
print(f"df_features: {df_features.shape[0]:,} rows x {df_features.shape[1]} cols")

if "also_buy" not in df_features.columns:
    raise KeyError(
        "df_features.pkl has no 'also_buy' column. The extraction ran against a "
        "CSV built before 'also_buy' was added to fields_to_keep in "
        "data/variable_selection.ipynb. Rebuild the CSV there, then re-run "
        "feature_extraction_workflow/extract_features.ipynb."
    )

print(f"\nunique asins   : {df_features['asin'].nunique():,}")
print(f"cat_3 values   : {df_features['cat_3'].nunique():,} (the schema categories)")
print(f"rows with features: {(df_features['extracted_features'].apply(len) > 0).sum():,}")
df_features[["asin", "cat_2", "cat_3", "cat_4", "also_buy", "extracted_features"]].head(3)

## 2. Clean `cat_4` against the reviewed taxonomy

`category_taxonomy.json` is a whitelist of the `(cat_3, cat_4)` pairs that are
real categories rather than product bullets that leaked into the category path
— `'Imported'`, `'10" high'`, `'measures 25x25x14cm'` and 550-odd others like
them. A value is kept only if it is valid **under its own parent**, since the
same label can be real in one branch and junk in another; everything else
becomes `<cat_3>_Other`.

The same function is applied to the catalogue lookup in §4, so both ends of an
edge are cleaned identically.

In [ ]:
with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

VALID_PAIRS = {
    (cat_3, value)
    for cat_2 in taxonomy
    for cat_3, values in taxonomy[cat_2].items()
    for value in values
}
MISSING = "Missing"
OTHER_SUFFIX = "_Other"

print(f"taxonomy: {len(taxonomy)} cat_2 | "
      f"{sum(len(v) for v in taxonomy.values())} cat_3 | "
      f"{len(VALID_PAIRS):,} valid (cat_3, cat_4) pairs")


def fold_cat_4(cat_3: pd.Series, cat_4: pd.Series) -> pd.Series:
    """Keep cat_4 where the (cat_3, cat_4) pair survived review, else <cat_3>_Other."""
    c3 = cat_3.astype(str)
    c4 = cat_4.fillna(MISSING).astype(str)
    keep = pd.Series(list(zip(c3, c4)), index=c3.index).isin(VALID_PAIRS)
    return pd.Series(np.where(keep, c4, c3 + OTHER_SUFFIX), index=c3.index)


df_features["cat_4_clean"] = fold_cat_4(df_features["cat_3"], df_features["cat_4"])

n_folded = int((df_features["cat_4_clean"] != df_features["cat_4"].fillna(MISSING)).sum())
print(f"\ndistinct cat_4: {df_features['cat_4'].nunique(dropna=False):,} -> "
      f"{df_features['cat_4_clean'].nunique():,}")
print(f"items folded into '<cat_3>{OTHER_SUFFIX}': {n_folded:,} "
      f"({n_folded / len(df_features):.2%})")

## 3. The base table — one row per product

`asin`, its cleaned category path, and its `also_buy` list. The raw column is a
stringified list (`"['B0001XR2F2', ...]"`), so it is parsed back to a real list
here; anything unparseable becomes an empty list rather than an error, and the
count below shows whether that is happening at any scale.

In [ ]:
def parse_asin_list(value) -> list:
    """Stringified list -> list of asins. Anything unparseable -> []."""
    if isinstance(value, list):
        return value
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []
    return parsed if isinstance(parsed, list) else []


df_base = df_features[["asin", "cat_2", "cat_3", "cat_4_clean"]].copy()
df_base["also_buy"] = df_features["also_buy"].apply(parse_asin_list)
df_base["also_buy_n"] = df_base["also_buy"].str.len()

with_edges = int((df_base["also_buy_n"] > 0).sum())
print(f"products                 : {len(df_base):,}")
print(f"with a non-empty also_buy: {with_edges:,} ({with_edges / len(df_base):.1%})")
print(f"co-purchase references   : {df_base['also_buy_n'].sum():,}")
print(f"per product, mean / max  : {df_base['also_buy_n'].mean():.1f} / "
      f"{df_base['also_buy_n'].max():,}")
print(f"median among non-empty   : "
      f"{df_base.loc[df_base['also_buy_n'] > 0, 'also_buy_n'].median():.0f}")

# extract_features.ipynb loads the CSV with .drop_duplicates(), so this should
# be 0. If it isn't, every edge from a repeated asin is counted twice in §5.
n_dupe_src = int(df_base["asin"].duplicated().sum())
if n_dupe_src:
    print()
    print(f"!! {n_dupe_src:,} duplicate source asins — §5 edges will double-count")

df_base[df_base["also_buy_n"] > 0].head(5)

## 4. Category lookup over the whole catalogue

`also_buy` targets are mostly *not* in `df_features` — that table stops at the
69 schema categories, while a co-purchase edge can point at any item. The
filtered CSV was never narrowed that way, so it is the lookup. Only `asin` and
`category` are read; the rest of the 2 GB stays on disk.

Items whose `cat_3` is outside the taxonomy get `<cat_3>_Other` from
`fold_cat_4`, which is the honest answer: unreviewed, so not trusted as a
leaf.

About a quarter of the CSV is exact full-row duplicates (60k sampled rows held
44,852 asins), which is why the dedupe count below is large — the same rows
`extract_features.ipynb` drops on load with `.drop_duplicates()`. They are byte
-identical repeats, so `keep="first"` discards no information.

The category path is parsed here rather than with
`feature_extraction_workflow.ensure_cat_columns`, which does the same thing:
importing that package pulls in `nltk` at module load, and the `RecSystem
(venv)` kernel does not have it. Nothing else in this notebook needs the
package, so it stays independent of that.

In [ ]:
def parse_category_levels(series: pd.Series, n_levels: int = 4) -> pd.DataFrame:
    """Stringified category path -> cat_1..cat_n columns (same rule as the pipeline)."""
    def as_list(value):
        if isinstance(value, list):
            return value
        if not isinstance(value, str) or not value.strip():
            return []
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
        return parsed if isinstance(parsed, list) else []

    paths = series.apply(as_list)
    return pd.DataFrame(
        {f"cat_{i + 1}": paths.apply(lambda p, i=i: p[i] if len(p) > i else None)
         for i in range(n_levels)},
        index=series.index,
    )


cat_lookup = pd.read_csv(
    CATALOGUE_PATH,
    usecols=["asin", "category"],
    low_memory=False,
)
print(f"catalogue rows: {len(cat_lookup):,}")

cat_lookup = pd.concat(
    [cat_lookup[["asin"]], parse_category_levels(cat_lookup["category"])], axis=1
)
cat_lookup["cat_4_clean"] = fold_cat_4(cat_lookup["cat_3"], cat_lookup["cat_4"])
cat_lookup = cat_lookup[["asin", "cat_2", "cat_3", "cat_4_clean"]]

n_dupes = int(cat_lookup["asin"].duplicated().sum())
cat_lookup = cat_lookup.drop_duplicates("asin", keep="first")
print(f"duplicate asins dropped: {n_dupes:,}")
print(f"lookup covers: {len(cat_lookup):,} asins "
      f"({len(cat_lookup) / len(df_base):.1f}x the {len(df_base):,} in df_features)")
cat_lookup.head(3)

## 5. Co-purchase pairs — the categories on both ends

One row per edge: the source product with its categories, the `also_buy`
target with its own. `~5M rows` at this grain, so the six category columns are
cast to `category` dtype — they hold a few hundred distinct values between
them and object strings at this length are what makes the table heavy.

`Not in catalogue` marks a target that appears in neither table. Those are real
edges — the co-purchase happened — pointing at books, electronics and anything
else Amazon sells, and the share of them is worth reading off the coverage
count below before aggregating.

In [ ]:
NOT_IN_CATALOGUE = "Not in catalogue"

pairs = (
    df_base.loc[df_base["also_buy_n"] > 0,
                ["asin", "cat_2", "cat_3", "cat_4_clean", "also_buy"]]
    .explode("also_buy", ignore_index=True)
    .rename(columns={
        "asin": "src_asin",
        "cat_2": "src_cat_2",
        "cat_3": "src_cat_3",
        "cat_4_clean": "src_cat_4",
        "also_buy": "dst_asin",
    })
)

pairs = pairs.merge(
    cat_lookup.rename(columns={
        "asin": "dst_asin",
        "cat_2": "dst_cat_2",
        "cat_3": "dst_cat_3",
        "cat_4_clean": "dst_cat_4",
    }),
    on="dst_asin",
    how="left",
)

resolved = pairs["dst_cat_2"].notna()
cat_cols = ["src_cat_2", "src_cat_3", "src_cat_4", "dst_cat_2", "dst_cat_3", "dst_cat_4"]
for col in ["dst_cat_2", "dst_cat_3", "dst_cat_4"]:
    pairs[col] = pairs[col].fillna(NOT_IN_CATALOGUE)
pairs[cat_cols] = pairs[cat_cols].astype("category")

print(f"co-purchase pairs   : {len(pairs):,}")
print(f"distinct source asins: {pairs['src_asin'].nunique():,}")
print(f"distinct target asins: {pairs['dst_asin'].nunique():,}")
print(f"targets resolved     : {resolved.sum():,} ({resolved.mean():.1%})")
print(f"'{NOT_IN_CATALOGUE}'  : {(~resolved).sum():,} ({(~resolved).mean():.1%})")
print(f"memory               : {pairs.memory_usage(deep=True).sum() / 1e9:.2f} GB")

pairs.head(10)